# Foundry voice sales from Fabric: DSVM quickstart

Choose this notebook **or** `Start-FoundryVoiceSales.ps1`. Both use the same workflow and backend; no duplicate agent implementation. Default **Run All makes no Azure calls, publishes nothing, installs nothing, and starts no server**. Azure results are **NOT RUN** in this template.

Read `readme.md` and `docs/access-and-networking.txt` first. Use a trusted single-user Windows DSVM session, Python 3.13, PowerShell 7.2+, Git and an existing VS Code Jupyter extension. To execute the Setup cell, first select an already working Python 3.13 kernel. If none exists, run `.\Start-FoundryVoiceSales.ps1 -Action Setup -Notebook` from PowerShell first, then select the template's `.venv` interpreter. Do not accept automatic global dependency installs or register a global kernel. Do not change system Python/Node/Azure settings.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

# If launched elsewhere, replace None with Path(r'<path-to-template>').
TEMPLATE_ROOT = None
ALLOW_AZURE = False
ALLOW_PUBLISH = False
RUN_SETUP = False
RUN_TESTS = False
RUN_SERVER = False
STOP_SERVER = False
REQUIRE_PRIVATE = False
PORT = 8765
QUESTION = 'Which approved sales measures can you answer from the configured sources?'

if TEMPLATE_ROOT is not None:
    ROOT = Path(TEMPLATE_ROOT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'Start-FoundryVoiceSales.ps1').is_file()), None)
if ROOT is None or not (ROOT / 'Start-FoundryVoiceSales.ps1').is_file():
    raise RuntimeError('Open this notebook from its template folder, or set TEMPLATE_ROOT to the copied template folder and rerun this cell.')
PWSH = shutil.which('pwsh')
if PWSH is None:
    raise RuntimeError('PowerShell 7.2+ is required. Ask your DSVM administrator to install it.')
ENV_FILE = ROOT / '.env'
PYTHON_EXECUTABLE = sys.executable
ONLINE_ACTIONS = {'Diagnose', 'Connectivity', 'Discover', 'Publish', 'Text', 'Smoke', 'Start'}

def run(action, *arguments):
    if action in ONLINE_ACTIONS and not ALLOW_AZURE:
        print(f'{action}: NOT RUN; set ALLOW_AZURE=True only after reviewing identity, resources, and access.')
        return
    if action == 'Publish' and not ALLOW_PUBLISH:
        print('Publish: NOT RUN; also set ALLOW_PUBLISH=True to create a Foundry prompt agent version.')
        return
    command = [PWSH, '-NoLogo', '-NoProfile', '-NonInteractive', '-File',
               str(ROOT / 'Start-FoundryVoiceSales.ps1'), '-Action', action,
               '-EnvFile', str(ENV_FILE)]
    if action in ONLINE_ACTIONS:
        command.append('-AllowAzure')
    command.extend(str(value) for value in arguments)
    return subprocess.run(command, cwd=ROOT, check=True)


## 1. Check local prerequisites (offline)
Checks tool availability only; it does not install tools, sign in, or verify Azure access.

In [ ]:
run('Check', '-PythonExecutable', PYTHON_EXECUTABLE)


## 2. Create the local environment (explicit package-download opt-in)
Using an already working Python 3.13 kernel, set `RUN_SETUP=True` and rerun the configuration cell, then this cell. This explicit notebook setup calls `Setup -Notebook`: it creates only the template `.venv` and installs `requirements-notebook.lock.txt`, including the base dependencies and `ipykernel`. Ordinary PowerShell `Setup` remains lightweight and installs only `requirements.lock.txt`. Package download may occur, but neither path accesses or provisions Azure resources. No global kernel is registered. After setup, select `.venv` as the VS Code notebook kernel and rerun configuration. Without an existing working kernel, run `.\Start-FoundryVoiceSales.ps1 -Action Setup -Notebook` in PowerShell instead; do not let VS Code auto-install packages globally. Copy `.env.example` to `.env` only if you do not already have `.env`, edit it privately, and never paste tokens or customer values into this notebook. See the README environment table.

In [ ]:
if RUN_SETUP:
    run('Setup', '-Notebook', '-PythonExecutable', PYTHON_EXECUTABLE)
else:
    print('Setup: NOT RUN; set RUN_SETUP=True to install optional notebook dependencies into the local .venv.')
if not ENV_FILE.is_file():
    print('Create .env from .env.example without overwriting an existing file, then fill approved values locally.')


## 3. Validate configuration and run offline tests
`Validate` is offline. Supply the existing published agent version; if you must publish a first version, follow the separately gated publication section below. A validation pass proves configuration syntax, not Azure permissions. Node is needed only for the optional offline UI tests; there are no frontend npm dependencies.

In [ ]:
venv_ready = (ROOT / '.venv' / 'Scripts' / 'python.exe').is_file() or (ROOT / '.venv' / 'bin' / 'python').is_file()
if ENV_FILE.is_file() and venv_ready:
    run('Validate')
else:
    print('Validate: NOT RUN; complete local Setup and prepare .env first.')
if RUN_TESTS:
    run('Test')


## 4. Review identity and networking, then opt in to Azure
Each DSVM OS user signs into Azure CLI as their own approved user in the configured tenant. In a separate terminal, use `az login --tenant '<tenant-id>'`; do not run `az account set`. The backend uses explicit tenant/subscription checks and that user's Fabric permissions. An Azure resource-group role does **not** grant Fabric/source access. Managed identity is unsupported for this Fabric path; service-principal preview is not selected.

Only after approval, set `ALLOW_AZURE=True` and rerun configuration. `Diagnose` requests identity/token diagnostics without printing tokens. `Connectivity` checks DNS/TLS; set `REQUIRE_PRIVATE=True` for private-routing checks. Private support remains conditional: see `docs/access-and-networking.txt`. Neither check proves data grounding.

In [ ]:
run('Diagnose')
run('Connectivity', *(['-RequirePrivate'] if REQUIRE_PRIVATE else []))
run('Discover')


## 5. Optional publisher-only step: create a persisted prompt agent version
**WRITE, disabled by default.** If an administrator already supplied `FOUNDRY_AGENT_NAME` and `FOUNDRY_AGENT_VERSION`, skip this. Otherwise the project publisher sets a verified existing `FOUNDRY_MODEL_DEPLOYMENT`, and both `ALLOW_AZURE=True` and `ALLOW_PUBLISH=True`. The Publish action creates only a Foundry `PromptAgentDefinition` version with `query_sales_data(question)`; it never provisions resources or deploys the local backend. Copy the returned version into `.env`, reset `ALLOW_PUBLISH=False`, and rerun Validate before text/start. Never make up an agent version to claim a successful validation.

In [ ]:
run('Publish')


## 6. Prove real text grounding first
Review `QUESTION` against the configured domain and approved data sources. Text/Smoke use the real Foundry prompt agent and Microsoft-hosted Fabric Data Agent MCP. Smoke combines diagnostics, discovery, and text; it does **not** test voice. A known-answer question must match a separately verified source result; do not accept a fluent ungrounded answer as proof.

In [ ]:
run('Smoke', '-Question', QUESTION)


## 7. Start the local browser experience
Set `RUN_SERVER=True` as well as `ALLOW_AZURE=True`, rerun configuration and this cell, then open the printed loopback URL. Start persists a PID/creation-time/executable/root ownership record and checks local HTTP health before returning. It does not publish an agent or deploy a server to Azure.

Text works with `VOICE_ENABLED=false`. For native live voice, first approve a compatible deployed model, regional processing scope, endpoint, voice, and pinned API version in `.env`. Windows Sound must show an actual input device; Edge needs microphone permission for the loopback page. RDP/browser microphone redirection is not automatic. If `getUserMedia` fails, use text; do not report audio success. Verify a spoken known-answer question, actual received playback, interruption/cancellation, and the authoritative grounded transcript.

In [ ]:
if RUN_SERVER:
    run('Start', '-Port', PORT)
else:
    print('Start: NOT RUN; set RUN_SERVER=True only when ready to open your local session.')


## 8. Shut down your local instance
Set `STOP_SERVER=True`, rerun configuration, then run this cell. Stop does not need `ALLOW_AZURE`, changes no cloud resources, and refuses mismatched/recycled process identities. A shared or untrusted multi-user host is unsupported: loopback binding is not OS-user authentication.

In [ ]:
if STOP_SERVER:
    run('Stop')
else:
    print('Stop: NOT RUN; set STOP_SERVER=True to stop only this template\'s owned instance.')


## Optional Fabric IQ extension: NOT RUN / NOT IMPLEMENTED
Fabric IQ is not part of this template, not a prerequisite, and not used for baseline sales answers. Any future IQ/ontology extension needs a separate reviewed implementation, permissions, endpoint contract, and private-network validation. There is deliberately no fake IQ code or installation cell.

Official references and architecture limits are in `docs/architecture.txt`. Clear all cell outputs before sharing or committing a locally executed notebook. Template verification is local only; Azure/Fabric/Voice Live/RDP tests are **NOT RUN**.